Colab setup : just download the jupyter and open it in your colab

# Self-Attention as a stack of Finite Dimensional Ising Models

## Introduction:

The exercise is divided into three parts: 
1) Ising model phase transition
2) Attention function
3) Mean field mapped into attention function 


## EXTRA: MACHINE EPSILON 
**What is the definition of machine epsilon? Does it depend on the precision?**
We talk a lot about precision in all of our lectures but, there is a limit and how o we find that limit is after gettign a understanding of our machine epsilion. Two points I would like you to take away:

- The largest relative spacing between any two adjacent numbers in the machine's floating point system. Yes, it depends on the system, so if its 64 bit IEEE floating point arithmetic, epsilon value is approx:  2.2204×10−16


- A measure of the precision **what is it** : what is the smallest number that can be added to 1.0 such that the sum differs from 1.0? That number is called the machine epsilon. Now because of these dependencies if I have bit size smaller, my significant digits numbers will be lesser as I will have less bits to allot to it, making my epsilon less precise.

In the following we are able to look at the epsilon behaviour
However, predict your answer before you run the cell!! 

In [ ]:
# Justification

import numpy as np
#A2 = np.float16 (1)
#A2 = np.float32 (1)
#A2=1  # this is also  np.float64(1)
while (A2/2.0 + 1.0)!= 1.0:
    A2 = A2 / 2.0
print (A2)
# alternate method create a 2-bit architecture then find epsilon in that vs the 64-bit you have on the laptop


0.000977


## 1. Ising model brute force 

To study the Ising model at a given temperature, we want to compute thermal averages - for example, the mean magnetization $\langle M \rangle$
<M>. In principle this means summing over every possible spin configuration, weighted by its Boltzmann probability P $\propto e^{-E/T}$

We use Monte carlo : which basically rather than summing everything, we **sample** configurations and average over those. The trick is to sample **smartly** because we want to spend more time in configurations that actually matter 
Hoeever rather than sampling independent configurations, we construct a random walk  type chain through configuration space where each step proposes a small change (flipping one spin) and accepts or rejects it via a critirion




In the Ising model, the energy between two spins $s_i$ and $s_j$ is:<br>
<br>
$$E_{ij} = -J \cdot s_i s_j$$
Where  J is the interaction energy / coupling energy 

The complete equation when the system is also being affected by external field<br> 
<br>
$$E_{ij} = -J \cdot s_i s_j + h \sum_i s_i$$

That external field also allows us to make mean-field solutions that then gives us magnetisation as a function of magnetisation itself ( sounds confusing but, its a whole derivation, let me know if you're interested to see it through) this then is plotted and allows us information on phase transition behaviour

Aligned spins ($s_i s_j > 0$) → negative energy → favoured by Boltzmann distribution. 

In the standard Ising model, magnetization is defined as:

$$
M = \frac{1}{N} \sum_i s_i
$$

- $s_i = \pm 1$ (spin at each lattice site)  
- $N = L \times L$ (total number of spins)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


L = 20              # lattice size (L x L)
n_steps = 5000         # Monte Carlo steps per temperature
T_vals = np.linspace(1.5, 4.0, 25)  # temperature range
Tc =              # known critical temperature for 2D Ising

# temperatures to check at look at what the limits above are
T_low = 
T_high = 


def initialize_lattice(L):
    return np.random.choice([-1, 1], size=(L, L))

def energy(lattice):
    E = 0
    for i in range(L):
        for j in range(L):
            S = lattice[i, j]
            neighbors = (
                lattice[(i)%L, j] + #neighbours are who? edit here in all four rows
                lattice[(i)%L, j] +
                lattice[i, (j)%L] +
                lattice[i, (j)%L]
            )
            E += -S * neighbors
    return E / 2

def magnetization(lattice):
    return np.abs(np.sum(lattice)) / (L*L)

#don't look here before you put in the neighbours above

#each step is proposing a small change (look at where the function is being called)
def metropolis_step(lattice, T):
    i = np.random.randint(0, L)
    j = np.random.randint(0, L)

    S = lattice[i, j]
    neighbors = (
        lattice[(i+1)%L, j] +
        lattice[(i-1)%L, j] +
        lattice[i, (j+1)%L] +
        lattice[i, (j-1)%L]
    )

    dE = 2 * S * neighbors

    if dE < 0 or np.random.rand() < np.exp(-dE / T): #smart decision making is happening here to spend more time in lower energy state 
        
        lattice[i, j] *= -1 #low energy flips always accepted

M_vs_T = []

lattice_low = None
lattice_high = None
lattice = initialize_lattice(L)

for T in T_vals:
    for _ in range(n_steps):
        metropolis_step(lattice, T) #lattice is passed every step and mutated allowing to build on previous 
    
    # for every T we want only its particular magnetisation
    Ms = []
    for _ in range(5000):
        metropolis_step(lattice, T)
        Ms.append(magnetization(lattice))

    M = np.mean(Ms)
    M_vs_T.append(M)
  
    #saving spin-configurations
    if abs(T - T_low) < 0.05:
        lattice_low = lattice.copy()
    if abs(T - T_high) < 0.05:
        lattice_high = lattice.copy()

#converting due to plotting needs in next step
M_vs_T = np.array(M_vs_T)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))

#what would you plot here?


# LEFT: 
#axes[0].imshow(, cmap="coolwarm")
axes[0].set_title()
axes[0].axis("off")

# MIDDLE
#axes[1].plot()
axes[1].axvline(Tc, linestyle="--", label="Tc")
#axes[1].set_xlabel("")
#axes[1].set_ylabel("")
axes[1].set_title("Phase Transition")
axes[1].legend()

# RIGHT
#axes[2].imshow(, cmap="coolwarm")
axes[2].set_title()
axes[2].axis("off")

plt.tight_layout()
plt.show()

EXTRA : Can you optimise this? How? 

## Answer to brute force:
Maybe the below helps you, but remember it is unoptimised still plots are UGLY!!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


L = 32              
n_steps = 20000         
T_vals = np.linspace(4.0, 1.5, 50)   #CAN BE OPTIMIZED TO FOCUS NEAR CRITICAL TEMPERATURE
Tc = 2.269            

T_low = 1.8
T_high = 3.5

def initialize_lattice(L):
    return np.random.choice([-1, 1], size=(L, L))

def energy(lattice):
    E = 0
    for i in range(L):
        for j in range(L):
            S = lattice[i, j]
            neighbors = (
                lattice[(i+1)%L, j] +
                lattice[(i-1)%L, j] +
                lattice[i, (j+1)%L] +
                lattice[i, (j-1)%L]
            )
            E += -S * neighbors
    return E / 2

def magnetization(lattice):
    return np.abs(np.sum(lattice)) / (L*L)


def metropolis_step(lattice, T):
    i = np.random.randint(0, L)
    j = np.random.randint(0, L)

    S = lattice[i, j]
    neighbors = (
        lattice[(i+1)%L, j] +
        lattice[(i-1)%L, j] +
        lattice[i, (j+1)%L] +
        lattice[i, (j-1)%L]
    )

    dE = 2 * S * neighbors

    if dE < 0 or np.random.rand() < np.exp(-dE / T):
        lattice[i, j] *= -1


M_vs_T = []

lattice_low = None
lattice_high = None
lattice = initialize_lattice(L)

for T in T_vals:

    for _ in range(n_steps):
        metropolis_step(lattice, T)
        
        
    Ms = []
    for _ in range(1000):
        metropolis_step(lattice, T)
        Ms.append(magnetization(lattice))

    M = np.mean(Ms)
    M_vs_T.append(M)
  
    if abs(T - T_low) < 0.05:
        lattice_low = lattice.copy()
    if abs(T - T_high) < 0.05:
        lattice_high = lattice.copy()

M_vs_T = np.array(M_vs_T)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(lattice_high, cmap="coolwarm")
axes[0].set_title(f"Disordered (T ≈ {T_high})")
axes[0].axis("off")

axes[1].plot(T_vals, M_vs_T)
axes[1].axvline(Tc, linestyle="--", label="Tc")
axes[1].set_xlabel("Temperature T")
axes[1].set_ylabel("|Magnetization|")
axes[1].set_title("Phase Transition")
axes[1].legend()

axes[2].imshow(lattice_low, cmap="coolwarm")
axes[2].set_title(f"Ordered (T ≈ {T_low})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

# 2. Ising model as attention activation

## 2.1 using cosine similarity

scroll two cells below if you don't know what cosine similarity means 

In [ ]:
import matplotlib.gridspec as gridspec     

## Define model base

In [ ]:
tokens  = ['The', 'cat', 'sat']# make your own token list keep 3 words if you are not planning to change embeddings at X below
seq_len = 3
d_model = 4

# alloted embeddings for a three listed token example
# there is something you will have to consider when setting the length of element in array
X = np.array([
    [1.0, 0.0, 1.0, 0.0],  
    [0.0, 2.0, 0.0, 2.0],  
    [1.0, 1.0, 1.0, 0.0],
    #
    #   
], dtype=np.float64)

print('Token embeddings X:')
for tok, vec in zip(tokens, X):
    print(f'  {tok:6s}: {vec}')

## Define Interaction energy
We model token embeddings as **spin vectors**. The interaction energy between token $i$ and token $j$ is:

$$E(i,j) = -J_{ij} = -\frac{x_i \cdot x_j}{\|x_i\| \|x_j\|}$$

RHS is simply the **negative cosine similarity**:
- Cosine = +1 (same direction / aligned spins) --> E = -1 → **very low energy** --> gets high attention
- Cosine =  0 (orthogonal / uncoupled spins) --> E =  0 → neutral
- Cosine = -1 (opposite / anti-aligned) --> E = +1 → **high energy** --> gets low attention



Question: Can you state the reason for assumption made here?   **Its answered already but, are you convinced?** 

Answer: 

In [ ]:
def ising_energy_matrix(X):
    # we normalise each row to get unit spin vectors

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm = X / (norms + 1e-8)         # shape: (seq_len, d_model)
    
    # similarity matrix: C[i,j] = x_i · x_j
    
    cosine_sim =                # shape: (seq_len, seq_len)  ??

    E =  #-J
    return E


E = ising_energy_matrix(X)


print('Ising energy matrix E(i,j)')
print(np.round(E, 4))
print('Interpretation: ')

Let's find where attention is being given through visualization

In [ ]:
# Energy landscape
sns.heatmap(E, annot=True, fmt='.3f', cmap='RdYlGn_r',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Ising Energy E(i,j) = -J\nGreen = low energy = will attend more', fontsize=11)
axes[1].set_xlabel('Token j'); axes[1].set_ylabel('Token i')
plt.show()

**ANSWER** for cosin'ing 

In [ ]:
def ising_energy_matrix(X):
 
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm = X / (norms + 1e-8)         
  
    cosine_sim = X_norm @ X_norm.T        
    
    E = -cosine_sim
    return E

E= ising_energy_matrix(X)

print('Ising energy matrix E(i,j) = -cosine_sim:')
print(np.round(E, 4))
print()


## 2.2 "non-cosine" method

Full Ising-model attention:
1. Compute pairwise energy E(i,j) via spin alignment (cosine)
2. Boltzmann factor: exp(-beta * E)
3. Normalise by partition function Z (= softmax equivalent)
4. Weighted sum of 'values' (raw embeddings X here)
**beta (inverse temperature)**
- all tokens get equal attention (uniform) when beta = 0
- only the lowest-energy token gets all attention when beta = large
- beta = 1 is sweet spot, constant scaling allowed with just $k_{B}$ dependence


In statistical physics, the **partition function** $Z$ is the sum over all states:
$$Z_i = \sum_j e^{-\beta E(i,j)}$$

Normalising gives the **Boltzmann probability distribution** - this IS our attention weight:
$$A(i,j) = \frac{e^{-\beta E(i,j)}}{Z_i}$$

In [ ]:
def boltzmann_weights(E, beta):
    E_stable = E - E.min(axis=1, keepdims=True)
    W = np.exp(-beta * E_stable)
    return W

beta = 5.0   # inverse temperature — try changing this!
W_unnorm = boltzmann_weights(E, beta)

In [ ]:
def ising_attention(X, beta):

    E, cosine_sim = ising_energy_matrix(X)
    neg_bE = -beta * E
   
    # ADD a line above to make stable bE!!
    W = np.exp(neg_bE_stable)
    
    # Partition function (denominator)
    Z = W.sum(axis=1, keepdims=True)
    
    # Normalised attention weights (Boltzmann distribution)
    A = W / Z   #rows sum to 1

    # Context output: weighted sum of value vectors (using X directly as V) in our case
    output = A @ X
    
    return A, output, E, Z

beta = 5.0
A_ising, output_ising, E_ising, Z_ising = ising_attention(X, beta=beta)

print(f'β = {beta}')
print('\nPartition function Z_i (normalisation constant per token):')
print(np.round(Z_ising.flatten(), 4))
print('\nIsing Attention Weights A(i,j) = exp(-βE) / Z:')
print(np.round(A_ising, 4))
#print('\nRow sums should be ??):', np.round(A_ising.sum(axis=1), 6))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(A_ising, annot=True, fmt='.3f', cmap='Reds',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title(f'Ising Attention Weights  (β={beta})\n'
             'A(i,j) = exp(-β·E(i,j)) / Z', fontsize=12)
ax.set_xlabel('Token j  (attended to)')
ax.set_ylabel('Token i  (attending)')
plt.tight_layout()
plt.show()

**Answer** non-cosining

In [ ]:
def ising_attention(X, beta):
    E, cosine_sim = ising_energy_matrix(X)
    neg_bE = -beta * E
    neg_bE_stable = neg_bE - neg_bE.max(axis=1, keepdims=True)  
    W = np.exp(neg_bE_stable)
   
    Z = W.sum(axis=1, keepdims=True)
    
    
    A = W / Z   # shape:rows sum to 1
    
    # Context output(using X directly as V)
    output = A @ X
    
    return A, output, E, Z

beta = 5.0
A_ising, output_ising, E_ising, Z_ising = ising_attention(X, beta=beta)

print(f'β = {beta}')
print('\nPartition function Z_i (normalisation constant per token):')
print(np.round(Z_ising.flatten(), 4))
print('\nIsing Attention Weights A(i,j) = exp(-βE) / Z:')
print(np.round(A_ising, 4))
print('\nRow sums (should be 1.0):', np.round(A_ising.sum(axis=1), 6))


### Mean field: 
replace all spin-spin interactions with a single average field $\langle s \rangle$ very different from **Original Ising:** where each spin $i$ must be updated by scanning all neighbours.
and 
$$\langle s \rangle = \sum_j P_j \, s_j = \sum_j \frac{e^{-\beta E_j}}{Z} s_j$$

Also here we will see $W_{K}$ and $W_{Q}$ are the global coupling matrices 

In [ ]:
tokens  = ['The', 'cat', 'sat']
#seq_len above
N       =        #seq_len above
d_model =   4     # spin dimensionality
d_k     =   4     # key/query dimension

# Token embeddings = spin vectors
X = np.array([
    [1.0, 0.0, 1.0, 0.0],   # 'The'  — spin s_0
    [0.0, 2.0, 0.0, 2.0],   # 'cat'  — spin s_1
    [1.0, 1.0, 1.0, 0.0],   # 'sat'  — spin s_2
], dtype=np.float64)

# Weight matrices (the global coupling structure)
np.random.seed(42)
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

# Project to get Q, K, V ( allow subspace representations to view data from different side ( this directionality is 
#is implicit. Why? because dot product hides cosine theta and theta is there))
#use @
Q =    # Queries  — local field each spin generates
K =    # Keys     — spin states in key-space
V =    # Values   — what each spin contributes to the output

print('Spin vectors (token embeddings X):')
for t, x in zip(tokens, X):
    print(f'  s_{tokens.index(t)} [{t:4s}] = {x}')
print()
print('Keys K (spin states in coupling space):')
print(np.round(K, 3))
print('\nQueries Q (local field each token generates):')
print(np.round(Q, 3))

How do we scale stuff in our interaction model? 
what is the attention headsize equivalence? 


$\frac{1}{\sqrt{d_{k}}}$ = $\frac{1}{k_{B}T}$ = $\beta$


In [ ]:
def mean_field_ising_attention(Q, K, V, beta=1.0):
    
    N, d_k = Q.shape

    # Ising coupling J[i,j] = Q_i · K_j
    J = Q @ K.T                         

    # Scale to set the effective temperature
    scaled_J = beta * J / np.sqrt(d_k)

    # Boltzmann weights = exp(β·J_ij) since E=-J 
    # Subtract row max for numerical stability (doesn't change result)
    scaled_J_stable = scaled_J - scaled_J.max(axis=1, keepdims=True)
    W = np.exp(scaled_J_stable)          # unnormalised Boltzmann factors

    # Partition function Z_i = Σ_j exp(β·J_ij) 
    Z = W.sum(axis=1, keepdims=True)     # (N, 1) - one scalar per spin

    #Mean-field probabilities (= attention weights)
    A = W / Z                            # (N, N), rows sum to 1 
    #remember W is scaled J and Z is also summed across N 

    mean_field = A @ V                   # (N, d_k)

    return A, mean_field, J, Z


beta = 1.0
A_mf, mf_output, J_mat, Z_mat = mean_field_ising_attention(Q, K, V, beta=beta)

print(f'β = {beta}  (inverse temperature)')
print('\nMean-field attention weights A[i,j] = exp(β·J_ij) / Z_i:')
for i, t in enumerate(tokens):
    row = ' '.join([f'{v:.4f}' for v in A_mf[i]])
    print(f'  Token "{t:3s}" → [{row}]  (sum={A_mf[i].sum():.6f})')
print()
print(f'Partition functions Z_i: {Z_mat.flatten().round(4)}')
print()
print("What did this mean field fix?")

Standard_attention!!
Can you write it yourself? 

In [ ]:
def standard_attention(Q, K, V):
    d_k = Q.shape[1]
    #
    #_stable = ------  - ------.max(axis=1, keepdims=True)
    W = np.exp()
    A = W / W.sum(axis=1, keepdims=True)
    return A, A @ V


A_std, std_output = standard_attention(Q, K, V)
A_mf1, mf_output1, _, _ = mean_field_ising_attention(Q, K, V, beta=1.0)

print('Standard attention weights:')
print(np.round(A_std, 6))
print()
print('Mean-field Ising attention weights (β=1):')
print(np.round(A_mf1, 6))
print()
max_diff = np.abs(A_std - A_mf1).max()
print(f'Max absolute difference: {max_diff:.2e}')
if max_diff < 1e-10:
    print('EXACT MATCH — mean-field Ising at β =1 IS standard self-attention.')
else:
    print(f'Difference: {max_diff} (should be ~0)')

print()
print('Output vectors match?')
print(f'  Max diff in output: {np.abs(std_output - mf_output1).max():.2e}')

In [ ]:
fig = plt.figure(figsize=(20, 9))
gs  = gridSpec(2, 5, figure=fig, hspace=0.5, wspace=0.4)

# Top row: standard attention (same for all β — shown once for reference)
ax_std = fig.add_subplot(gs[0, 0])
sns.heatmap(A_std, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=ax_std, vmin=0, vmax=1, cbar=False)
ax_std.set_title('Standard\nSoftmax Attention\n(β=1, reference)', fontsize=10)
ax_std.set_xlabel('Key j'); ax_std.set_ylabel('Query i')

# Add arrow
fig.text(0.265, 0.75, '<- same as -> ', ha='center', va='center',
         fontsize=11, color='navy', style='italic')

# Top row: mean-field Ising at β=1
ax_mf1 = fig.add_subplot(gs[0, 1])
A_tmp, _, _, _ = mean_field_ising_attention(Q, K, V, beta=1.0)
sns.heatmap(A_tmp, annot=True, fmt='.3f', cmap='Reds',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=ax_mf1, vmin=0, vmax=1, cbar=False)
ax_mf1.set_title('Mean-Field Ising\nβ=1', fontsize=10)


In [ ]:
fig.suptitle('Mean-Field Ising Attention compared to Attention function ',fontsize=13)
plt.show()

**Answer**
I KNOW its more theoretical this exercise and is different but...try peeking then work again on exercise 
However if not, then go through the partial solution!


In [ ]:
tokens  = ['The', 'cat', 'sat']
N       = 3       
d_model = 4       
d_k     = 4       


X = np.array([
    [1.0, 0.0, 1.0, 0.0], [0.0, 2.0, 0.0, 2.0], [1.0, 1.0, 1.0, 0.0],], dtype=np.float64)

np.random.seed(42)
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

# Project to get Q, K, V
Q = X @ W_Q   
K = X @ W_K   
V = X @ W_V   

print('Spin vectors (token embeddings X):')
for t, x in zip(tokens, X):
    print(f'  s_{tokens.index(t)} [{t:4s}] = {x}')
print()
print('Keys K (spin states in coupling space):')
print(np.round(K, 3))
print('\nQueries Q (local field each token generates):')
print(np.round(Q, 3))


def mean_field_ising_attention(Q, K, V, beta=1.0):
    
    N, d_k = Q.shape
    J = Q @ K.T                       

    scaled_J = beta * J / np.sqrt(d_k)
    scaled_J_stable = scaled_J - scaled_J.max(axis=1, keepdims=True)
    W = np.exp(scaled_J_stable)          
    Z = W.sum(axis=1, keepdims=True)     # one scalar per spin

    A = W / Z                            # (N, N), rows sum to 1
    mean_field = A @ V                   # (N, d_k)

    return A, mean_field, J, Z


beta = 1.0
A_mf, mf_output, J_mat, Z_mat = mean_field_ising_attention(Q, K, V, beta=beta)


for i, t in enumerate(tokens):
    row = ' '.join([f'{v:.4f}' for v in A_mf[i]])
    print(f'  Token "{t:3s}" -> [{row}]  (sum={A_mf[i].sum():.6f})')
print()
print(f'Partition functions Z_i: {Z_mat.flatten().round(4)}')



def standard_attention(Q, K, V):
    d_k = Q.shape[1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores_stable = scores - scores.max(axis=1, keepdims=True)
    W = np.exp(scores_stable)
    A = W / W.sum(axis=1, keepdims=True)
    return A, A @ V